# RAG for Image Captioning - Technical Report

## Project overview

This project builds a retrieval-augmented generation (RAG) pipeline for image captioning using the MS COCO dataset. The system combines image-level metadata, caption text, and retrieval indexing to support caption generation and evaluation.

The main goal is not only to generate captions, but also to preserve a clear distinction between:
- raw language data from COCO captions
- cleaned caption data produced by the preprocessing pipeline
- visual metadata generated from YOLO-World detection results
- final validation checks before using the retrieval system

This distinction is important because visual statistics and language statistics must be analyzed from different data sources.

## 1. Problem definition

Image captioning models may generate captions that are syntactically fluent but semantically inaccurate. Typical issues include:
- missing objects in the image
- hallucinated objects or attributes
- weak modeling of object relationships and scene context
- poor grounding with the visual structure of the image

The project addresses this by building a retrieval-based augmentation setup. Before generation, the model can retrieve similar image-caption examples, which provide additional scene information and reduce hallucination risk.

## 2. Dataset and source files

The project uses Microsoft COCO 2017. The main dataset is organized under the `data/` folder.

### Input dataset files
- `data/annotations_trainval2017/annotations/captions_train2017.json`
- `data/annotations_trainval2017/annotations/captions_val2017.json`
- `data/annotations_trainval2017/annotations/instances_train2017.json`
- `data/annotations_trainval2017/annotations/instances_val2017.json`
- `data/val2017/` for validation images
- `data/train2017_subset/` for the downloaded training subset

### Split files produced by the project
- `data/coco_rag_data/train_kb_ids.json`
- `data/coco_rag_data/val_ids.json`
- `data/coco_rag_data/test_ids.json`

These split files are the source of truth for anti-leakage and dataset partitioning. They ensure that the knowledge base does not contain validation or test images.

## 3. Data source separation

One of the most important aspects of this project is correctly separating the data sources used for statistics.

### 3.1 Caption corpus
The caption corpus comes from COCO captions and represents the language side of the dataset.
- raw captions are used for original caption statistics
- cleaned captions are used for preprocessing comparisons

### 3.2 Preprocessed caption store
The cleaned caption corpus is generated by `scripts/caption_pipeline.py` and stored as:
- `data/coco_rag_data/captions_kb.json`
- `data/coco_rag_data/captions_kb_flat.jsonl`

Each entry contains:
- `caption_raw`
- `caption_clean`

This is the correct source for caption preprocessing statistics, vocabulary comparison, and caption-length analysis after cleaning.

### 3.3 Unified visual lookup
Visual statistics should come from the unified lookup, not from raw COCO instance annotation files. The lookup is created in `scripts/build_unified_lookup.py` and stored as:
- `data/coco_rag_data/train_kb_lookup.json`
- `data/coco_rag_data/test_lookup.json`
- `data/coco_rag_data/val_lookup.json`

The lookup stores per-image metadata including:
- `objects`
- `attributes`
- `relationships`
- `scene_context`

These fields are used for object frequency analysis, spatial and visual relationship analysis, and scene complexity.

## 4. Project pipeline

The overall pipeline is organized in the following order:

1. Download the COCO subset and metadata
2. Create split files with anti-leakage constraints
3. Build the unified lookup for visual metadata
4. Clean and normalize captions
5. Extract text embeddings and image embeddings
6. Build FAISS indexes
7. Validate the pipeline and retrieval outputs
8. Run final evaluation

This structure ensures that all retrieval and evaluation stages are reproducible and consistent with the dataset splits.

## 5. Required inputs and generated outputs

### Required inputs
These files are required before running the pipeline:
- `data/annotations_trainval2017/annotations/captions_train2017.json`
- `data/annotations_trainval2017/annotations/captions_val2017.json`
- `data/annotations_trainval2017/annotations/instances_train2017.json`
- `data/annotations_trainval2017/annotations/instances_val2017.json`
- `data/val2017/`
- `data/train2017_subset/`

### Generated outputs
The project generates the following output artifacts:
- `data/coco_rag_data/train_kb_ids.json`
- `data/coco_rag_data/val_ids.json`
- `data/coco_rag_data/test_ids.json`
- `data/coco_rag_data/train_kb_lookup.json`
- `data/coco_rag_data/captions_kb.json`
- `data/coco_rag_data/captions_kb_flat.jsonl`
- `data/coco_rag_data/text_embeddings.npy`
- `data/coco_rag_data/text_index.faiss`
- `data/coco_rag_data/image_embeddings/`
- `data/coco_rag_data/image_index.faiss`
- `data/coco_rag_data/phase8_quality_control_report.json`

These files are the concrete outputs of the pipeline and should be treated as the authoritative artifact set for later analysis.

## 6. File responsibilities by stage

### Stage 1: Data preparation
- `scripts/download_coco_rag_subset.py` — downloads the selected COCO subset
- `scripts/create_splits.py` — creates split files
- `scripts/diagnose_images.py` — checks for missing or corrupted images

### Stage 2: Metadata and lookup construction
- `scripts/build_unified_lookup.py` — builds image-level metadata and object/relationship dictionaries
- `data/coco_rag_data/train_kb_lookup.json` — used for scene statistics and object counts

### Stage 3: Caption preprocessing
- `scripts/caption_pipeline.py` — normalizes raw captions and creates `caption_clean`
- `data/coco_rag_data/captions_kb.json` — cleaned caption store

### Stage 4: Embeddings and retrieval
- `scripts/extract_image_embeddings.py` — image embeddings
- `scripts/extract_text_embeddings.py` — text embeddings
- `scripts/build_faiss_index.py` — vector index creation

### Stage 5: Quality assurance
- `scripts/phase8_quality_control.py` — checks the integrity of the retrieval pipeline
- `data/coco_rag_data/phase8_quality_control_report.json` — final validation report

## 7. Recommended analysis logic

The correct analysis logic is the following:

- Use raw COCO captions for original caption-language statistics
- Use `caption_clean` for preprocessing and cleaned-caption statistics
- Use `lookup.objects` for object frequency and scene complexity analysis
- Use `lookup.relationships` only for visual relationship analysis
- Keep caption-based spatial relations and image-based visual relations in separate sections
- Do not compare raw textual statistics with YOLO-derived detector metadata in the same metric group

This prevents misleading interpretations such as using detector objects as if they were a text vocabulary.

## 8. Quality assurance and validation

The project includes a final quality control stage to ensure the retrieval system is valid before downstream evaluation.

The script `scripts/phase8_quality_control.py` checks:
- anti-leakage between train, validation, and test splits
- consistent image and caption mapping
- text embedding integrity
- FAISS index integrity
- alignment between image lookup data and embedding vectors
- simple self-retrieval sanity tests

This is the final gate before using the knowledge base in retrieval-based caption generation.

## 9. Example workflow

```bash
python scripts/create_splits.py
python scripts/build_unified_lookup.py
python scripts/caption_pipeline.py --input data/coco_rag_data/train_kb_lookup.json --output-dir data/coco_rag_data
python scripts/extract_text_embeddings.py
python scripts/extract_image_embeddings.py
python scripts/build_faiss_index.py
python scripts/phase8_quality_control.py
```

These commands should be executed in order, because each step produces the input required by the next stage.

In [ ]:
import json
from pathlib import Path

paths = [
    Path('data/annotations_trainval2017/annotations/captions_train2017.json'),
    Path('data/coco_rag_data/train_kb_ids.json'),
    Path('data/coco_rag_data/train_kb_lookup.json'),
    Path('data/coco_rag_data/captions_kb.json'),
    Path('data/coco_rag_data/text_embeddings.npy'),
    Path('data/coco_rag_data/text_index.faiss'),
]

for p in paths:
    print(f'{p}:', 'OK' if p.exists() else 'MISSING')

## 10. Final conclusion

The project is best described as a structured multimodal retrieval pipeline.

The most important design principle is that the project must maintain a clear separation between:
- language data
- visual data
- metadata generated from image detection
- validation checks for retrieval reliability

This separation is essential for correct analysis, correct reporting, and trustworthy evaluation.